# Adaptive window-parallel maximum entropy knockoffs

This notebook walks through the approximate-independence idea from `main.tex` using the maximum entropy solver. If the covariance is locally structured, sparse Cholesky updates in different windows can have negligible overlap. The package treats this as a scheduling heuristic: it chooses weak cross-correlation boundaries, runs serial coordinate descent inside each window, runs windows concurrently, and validates the maintained Cholesky factor after optimization.

For a real parallel run, start Julia with multiple threads before opening this notebook, for example:

```bash
JULIA_NUM_THREADS=4 julia --project
```


In [5]:
using Knockoffs
using LinearAlgebra
using Random
using Statistics
using Printf

BLAS.set_num_threads(1)
println("Julia threads available to the parallel solver: ", Threads.nthreads())

Julia threads available to the parallel solver: 4


## 1. Simulate locally structured model-X data

The example below creates four independent AR(1)-like covariance blocks, then randomly permutes the variables. The covariance still has local structure, but it is hidden from the input order. This lets the package's automatic nearest-correlation ordering visibly reorder the features before it chooses approximately independent windows.

In [6]:
Random.seed!(2026)

function block_ar_cov(p::Int; blocksize::Int=30, rhos=[0.72, 0.55, 0.82, 0.35])
    Σ = Matrix{Float64}(I, p, p)
    for (b, lo) in enumerate(1:blocksize:p)
        hi = min(p, lo + blocksize - 1)
        rho = rhos[mod1(b, length(rhos))]
        for j in lo:hi, i in (j + 1):hi
            Σ[i, j] = rho^(i - j)
            Σ[j, i] = Σ[i, j]
        end
    end
    return Symmetric(Σ, :U)
end

n = 250
p = 120
m = 1

Σlocal = block_ar_cov(p)
input_permutation = shuffle(1:p)
Σ = Symmetric(Matrix(Σlocal)[input_permutation, input_permutation], :U)
X = randn(n, p) * cholesky(Σ).U

println("Simulated X with size ", size(X), " and covariance size ", size(Σ), ".")
println("The covariance was randomly permuted, so automatic feature re-ordering should occur.")
@printf("Smallest eigenvalue of Σ: %.3e\n", eigmin(Σ))
@printf("Largest absolute off-diagonal correlation: %.3f\n", maximum(abs, Matrix(Σ - I)))

Simulated X with size (250, 120) and covariance size (120, 120).
The covariance was randomly permuted, so automatic feature re-ordering should occur.
Smallest eigenvalue of Σ: 9.917e-02
Largest absolute off-diagonal correlation: 0.820


## 2. Run the package solver and read its diagnostics

The useful intermediate output comes from the package itself. With `verbose=true`, `method=:maxent_fast` reports whether feature re-ordering occurred, how many approximately independent windows were found, how many Julia threads and worker threads are being used, and then prints one line per coordinate-descent sweep.

Here we intentionally omit `feature_order`, so the solver computes its own locality-promoting order from nearest-correlation neighbors.

In [7]:
λmin = 1e-6
solver_workers = min(2, Threads.nthreads())

s_fast = solve_s(Symmetric(Σ), :maxent_fast;
    m=m,
    niter=30,
    tol=1e-5,
    λmin=λmin,
    verbose=true,
    nworkers=solver_workers,
    boundary_band=8,
    min_window_size=20,
    window_corr_tol=1e-4,
    factor_check=true,
    factor_check_tol=1e-3)

@printf("\nFinished maximum entropy optimization. mean(s) = %.4f, min(s) = %.4e, max(s) = %.4f\n",
    mean(s_fast), minimum(s_fast), maximum(s_fast))

Adaptive window-parallel setup:
  Feature ordering: automatic nearest-correlation ordering; features were reordered.
  Approximately independent windows: 2
  Julia threads available: 4
  Worker threads used: 2
  Boundary band: 8
  Minimum window size: 20
  Window correlation tolerance: 0.0001
  Window ranges in ordered coordinates: 1:30, 31:120
  Window sizes: 30, 90
Iter 1: δ = 0.6652824084984044, windows = 2
Iter 2: δ = 0.09197992331808691, windows = 2
Iter 3: δ = 0.0067887391879124515, windows = 2
Iter 4: δ = 0.0015500845765733517, windows = 2
Iter 5: δ = 0.00022738009770217893, windows = 2
Iter 6: δ = 5.0821268333445246e-5, windows = 2
Iter 7: δ = 8.281524283226815e-6, windows = 2
Post-optimization Cholesky check: passed.
  Verified L'L - lambda_min*I ≈ ((m + 1) / m)Σ - S.
  Relative residual: 7.104058806445455e-15
  Tolerance: 0.001

Finished maximum entropy optimization. mean(s) = 0.3819, min(s) = 1.3622e-01, max(s) = 0.8078
